# StreamSentinel · API d'analyse en direct

Ce notebook réunit le pipeline **J4 (version corrigée au J6) + J5** dans une seule
fonction `analyse_complete(photo)`, puis l'expose sur le web pour l'interface Streamlit.

**Mode d'emploi**
1. Activer le **GPU T4** (Exécution → Modifier le type d'exécution).
2. **Exécution → Tout exécuter**. Compter 5 à 10 minutes pour le chargement des modèles.
3. La dernière cellule affiche une adresse `https://….trycloudflare.com` : la copier dans l'app Streamlit.
4. Laisser l'onglet Colab ouvert pendant la démo : si la session s'arrête, l'API s'arrête aussi.

Changements par rapport aux notebooks J4 et J5 :
- `normaliser_vlm` traite `NOT_SURE` comme une réponse valide (correction du J6 qui manquait ici) ;
- l'adresse Hub'Eau s'appelle `HUBEAU_URL` (elle écrasait `BASE`, le chemin du Drive) ;
- les photos reçues et leurs images annotées sont rangées dans `/content/api/`, pas sur le Drive.


## 0. Installation, GPU, Drive

In [1]:
!pip install -q transformers accelerate ultralytics fastapi uvicorn python-multipart

import torch, os, glob, json, re, base64, time, threading, uuid
import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Appareil utilisé :", device)

from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/streamsentinel"
DOSSIERS = {k: f"{BASE}/{k}" for k in ["photos", "resultats", "modeles", "annotees"]}
for d in DOSSIERS.values():
    os.makedirs(d, exist_ok=True)

EXTENSIONS = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
photos = sorted(p for ext in EXTENSIONS for p in glob.glob(f"{DOSSIERS['photos']}/{ext}"))
print(len(photos), "photos")

TAILLE_MAX = 1024

def charger(chemin):
    img = ImageOps.exif_transpose(Image.open(chemin)).convert("RGB")
    img.thumbnail((TAILLE_MAX, TAILLE_MAX))
    return img

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 8.5 MB/s eta 0:00:00
Appareil utilisé : cuda
Mounted at /content/drive
50 photos


## 1. Le formulaire OneAquaHealth

In [2]:
FORMULAIRE = {
    # étape 4 — vue sur 100 m
    "channel_form":     {"etape": 4, "options": ["FLAT", "U_SHAPE", "V_SHAPE"], "source": "vlm"},
    "bottom_type":      {"etape": 4, "options": ["NAT", "ARTIFICIAL"], "source": "vlm"},
    "bank_type":        {"etape": 4, "options": ["NAT", "ARTIFICIAL", "LAYED_STONES"], "source": "regle+vlm"},
    "water_flow":       {"etape": 4, "options": ["FAS", "SLOW", "STAGNANT", "DRY"], "source": "vlm"},
    # étape 5
    "water_aspect":     {"etape": 5, "options": ["CLEAR", "MUDDY", "FOAM", "ALTERED_COLOR"], "source": "regle"},
    "barriers":         {"etape": 5, "options": ["YES", "NO"], "source": "vlm"},
    "draining_pipes":   {"etape": 5, "options": ["YES", "NO"], "source": "vlm"},
    "sewage_discharge": {"etape": 5, "options": ["YES", "NO"], "source": "vlm"},
    "construction":     {"etape": 5, "options": ["YES", "NO"], "source": "vlm"},
    # étape 6 — berges sur 5 à 10 m
    "impervious_left":       {"etape": 6, "options": ["YES", "NO"], "source": "regle"},
    "impervious_right":      {"etape": 6, "options": ["YES", "NO"], "source": "regle"},
    "vegetation_left":       {"etape": 6, "options": ["YES", "NO"], "source": "regle"},
    "vegetation_right":      {"etape": 6, "options": ["YES", "NO"], "source": "regle"},
    "vegetation_type_left":  {"etape": 6, "options": ["HERBS", "SHRUBS", "TREES"], "source": "regle"},
    "vegetation_type_right": {"etape": 6, "options": ["HERBS", "SHRUBS", "TREES"], "source": "regle"},
    "vegetation_cuts":       {"etape": 6, "options": ["YES", "NO"], "source": "vlm"},
}

# laissés au citoyen, et pourquoi
LAISSES_AU_CITOYEN = {
    "research_site": "choix du site officiel",
    "water_height": "mesure physique, impossible sur photo",
    "water_withdrawal": "pompage rarement visible",
    "habitats": "sous-options non relevées dans l'app",
    "natural_debris": "sous-options non relevées dans l'app",
    "invasive_species": "liste d'espèces non relevée dans l'app",
    "feelings": "ressenti personnel : l'IA ne le remplit jamais",
}

SEUIL_CONFIANCE = 0.6   # en dessous : NOT_SURE
print(len(FORMULAIRE), "champs pré-remplis par l'IA,", len(LAISSES_AU_CITOYEN), "laissés au citoyen")

16 champs pré-remplis par l'IA, 7 laissés au citoyen


## 2. Segmentation (SegFormer)

In [3]:
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

SEG_ID = "nvidia/segformer-b2-finetuned-ade-512-512"
seg_processor = SegformerImageProcessor.from_pretrained(SEG_ID)
seg_model = SegformerForSemanticSegmentation.from_pretrained(SEG_ID).to(device).eval()

GROUPES = {
    1: ("eau",         ["water", "sea", "river", "lake", "waterfall"]),
    2: ("arbres",      ["tree", "palm"]),
    3: ("herbes",      ["grass", "field", "flower"]),
    4: ("arbustes",    ["plant"]),
    5: ("impermeable", ["building", "wall", "house", "road", "sidewalk", "bridge", "skyscraper",
                        "fence", "railing", "column", "stairs", "pier", "car", "bench", "path"]),
    6: ("sol_naturel", ["earth", "rock", "sand", "mountain", "hill", "dirt track"]),
    7: ("ciel",        ["sky"]),
}
EAU, ARBRES, HERBES, ARBUSTES, IMPER, SOL, CIEL = 1, 2, 3, 4, 5, 6, 7
VEGETATION = [ARBRES, HERBES, ARBUSTES]

table = np.zeros(len(seg_model.config.id2label), dtype=np.uint8)
for idx, nom in seg_model.config.id2label.items():
    court = nom.split(";")[0].strip().lower()
    for g, (_, classes) in GROUPES.items():
        if court in classes:
            table[int(idx)] = g

def nettoyer_eau(masque, part_min=0.01):
    eau = (masque == EAU).astype(np.uint8)
    eau = cv2.morphologyEx(eau, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
    n, etiquettes, stats, _ = cv2.connectedComponentsWithStats(eau)
    garder = np.zeros_like(eau)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= part_min * masque.size:
            garder[etiquettes == i] = 1
    propre = masque.copy()
    propre[(masque == EAU) & (garder == 0)] = 0
    propre[(garder == 1) & (masque != EAU)] = EAU
    return propre

@torch.no_grad()
def segmenter(img_pil):
    entrees = seg_processor(images=img_pil, return_tensors="pt").to(device)
    seg = seg_processor.post_process_semantic_segmentation(
        seg_model(**entrees), target_sizes=[img_pil.size[::-1]])[0]
    return nettoyer_eau(table[seg.cpu().numpy().astype(np.int64)])

preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/6.88k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  110MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  110MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

## 3. Analyse de l'eau

In [4]:
def analyser_eau(img_pil, masque):
    eau = (masque == EAU)
    if eau.sum() < 0.05 * masque.size:
        return None
    rgb = np.array(img_pil)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    h, s, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    gris = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    n = eau.sum()

    texture_locale = cv2.blur(np.abs(cv2.Laplacian(gris, cv2.CV_64F)), (15, 15))
    blanc = (eau & (s < 60) & (v > 175) & (texture_locale < 25)).astype(np.uint8)
    blanc = cv2.morphologyEx(blanc, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    part_mousse = blanc.sum() / n

    part_vert = (eau & (h >= 30) & (h <= 90) & (s > 50) & (v > 40)).sum() / n
    texture_eau = float(np.abs(cv2.Laplacian(gris, cv2.CV_64F))[eau].mean())

    part_brun = (eau & (h >= 8) & (h <= 30) & (s > 50)).sum() / n
    score_turb = float(np.clip(0.7 * part_brun + 0.3 * (1 - min(texture_eau / 20, 1)), 0, 1))

    return {"mousse": float(part_mousse), "vert": float(part_vert),
            "eau_lisse": texture_eau < 25, "turbidite": score_turb}

## 4. Règles berges et eau

In [5]:
def reponse(valeur, confiance, raison, source="regle"):
    return {"valeur": valeur, "confiance": round(float(confiance), 2), "raison": raison, "source": source}

def pas_sur(raison, source="regle"):
    return reponse("NOT_SURE", 0.0, raison, source)

def marges(masque, epaisseur=0.06):
    # anneau de pixels autour de l'eau, découpé en moitié gauche et droite de l'image
    eau = (masque == EAU).astype(np.uint8)
    k = max(5, int(masque.shape[1] * epaisseur))
    anneau = (cv2.dilate(eau, np.ones((k, k), np.uint8)) - eau).astype(bool)
    anneau &= (masque != CIEL)
    milieu = masque.shape[1] // 2
    gauche, droite = anneau.copy(), anneau.copy()
    gauche[:, milieu:] = False
    droite[:, :milieu] = False
    return {"image_gauche": masque[gauche], "image_droite": masque[droite], "tout": masque[anneau]}

def regles_berges(masque, orientation="aval"):
    m = marges(masque)
    # conversion image -> rive, selon le sens de la photo
    if orientation == "aval":
        rives = {"left": m["image_gauche"], "right": m["image_droite"]}
    else:
        rives = {"left": m["image_droite"], "right": m["image_gauche"]}

    champs = {}
    for cote, pix in rives.items():
        if pix.size < 500:
            for nom in ["impervious", "vegetation", "vegetation_type"]:
                champs[f"{nom}_{cote}"] = pas_sur("rive trop peu visible")
            continue

        part_imper = float(np.isin(pix, [IMPER]).mean())
        part_veg = float(np.isin(pix, VEGETATION).mean())

        # surfaces imperméables : question « plus d'un tiers de la marge ? »
        if part_imper > 0.40:
            champs[f"impervious_{cote}"] = reponse("YES", min(1, 0.6 + part_imper / 2), f"{part_imper:.0%} de surfaces artificielles")
        elif part_imper < 0.20:
            champs[f"impervious_{cote}"] = reponse("NO", min(1, 0.6 + (0.2 - part_imper) * 2), f"{part_imper:.0%} de surfaces artificielles")
        else:
            champs[f"impervious_{cote}"] = pas_sur(f"{part_imper:.0%}, proche du seuil d'un tiers")

        # présence de végétation
        if part_veg > 0.25:
            champs[f"vegetation_{cote}"] = reponse("YES", min(1, 0.6 + part_veg / 2), f"{part_veg:.0%} de végétation")
        elif part_veg < 0.08:
            champs[f"vegetation_{cote}"] = reponse("NO", 0.7, f"{part_veg:.0%} de végétation")
        else:
            champs[f"vegetation_{cote}"] = pas_sur(f"{part_veg:.0%} de végétation, cas limite")

        # type dominant : doit dépasser 50 % de la végétation
        veg = pix[np.isin(pix, VEGETATION)]
        if veg.size > 300:
            parts = {code: float((veg == g).mean()) for code, g in
                     [("TREES", ARBRES), ("HERBS", HERBES), ("SHRUBS", ARBUSTES)]}
            dominant, part = max(parts.items(), key=lambda x: x[1])
            if part > 0.5:
                champs[f"vegetation_type_{cote}"] = reponse(dominant, 0.5 + part / 2, f"{part:.0%} de la végétation")
            else:
                champs[f"vegetation_type_{cote}"] = pas_sur("aucun étage ne dépasse 50 %")
        else:
            champs[f"vegetation_type_{cote}"] = pas_sur("pas assez de végétation visible")

    # type de berge (les deux rives confondues)
    tout = m["tout"]
    if tout.size > 500:
        part_imper = float((tout == IMPER).mean())
        if part_imper > 0.6:
            champs["bank_type"] = reponse("ARTIFICIAL", 0.5 + part_imper / 2, f"{part_imper:.0%} de berge artificielle")
        elif part_imper < 0.3:
            champs["bank_type"] = reponse("NAT", 0.5 + (1 - part_imper) / 2, f"{part_imper:.0%} de berge artificielle")
        else:
            champs["bank_type"] = pas_sur(f"berge mixte ({part_imper:.0%} artificielle)")
    return champs

def regle_eau(analyse):
    # choix unique dans l'app : priorité mousse > couleur anormale > boue > claire
    if analyse is None:
        return pas_sur("eau pas assez visible")
    if analyse["mousse"] > 0.14:
        return reponse("FOAM", min(1, 0.6 + analyse["mousse"]), f"mousse sur {analyse['mousse']:.0%} de l'eau")
    if analyse["vert"] > 0.18 and analyse["eau_lisse"]:
        return reponse("ALTERED_COLOR", min(1, 0.55 + analyse["vert"]), f"eau verte sur {analyse['vert']:.0%} de la surface")
    if analyse["turbidite"] > 0.35:
        return reponse("MUDDY", min(1 , 0.5 + analyse["turbidite"] / 2), f"turbidité {analyse['turbidite']:.2f}")
    if analyse["turbidite"] < 0.30:
        return reponse("CLEAR", 0.85 - analyse["turbidite"], f"turbidité {analyse['turbidite']:.2f}")
    return pas_sur(f"turbidité intermédiaire ({analyse['turbidite']:.2f})")

## 5. Modèle vision-langage (Qwen2.5-VL-3B)

In [6]:
from transformers import AutoProcessor, AutoModelForImageTextToText

VLM_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
DTYPE = torch.float16

vlm_processor = AutoProcessor.from_pretrained(VLM_ID, min_pixels=256 * 28 * 28, max_pixels=640 * 28 * 28)
vlm = AutoModelForImageTextToText.from_pretrained(VLM_ID, torch_dtype=DTYPE, device_map="auto").eval()
print("Modèle vision-langage chargé")

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Modèle vision-langage chargé


In [7]:
QUESTIONS_VLM = {
    "channel_form": "Cross-section shape of the stream channel. FLAT = wide and flat bottom, "
                    "U_SHAPE = rounded with steep banks, V_SHAPE = narrow and deep like a V.",
    "bottom_type": "Stream bed material. NAT = natural (gravel, stones, sand, mud), "
                   "ARTIFICIAL = concrete or paved bottom. Answer NOT_SURE if the bottom is not visible.",
    "bank_type": "Banks. NAT = natural soil and plants, ARTIFICIAL = concrete walls, "
                 "LAYED_STONES = placed stones or riprap without concrete.",
    "water_flow": "Water flow. FAS = fast, SLOW = slow, STAGNANT = standing water, DRY = no water. "
                  "Only answer if clearly visible from ripples or still surface.",
    "barriers": "Is there a weir, dam, sluice or other barrier across the stream? YES or NO.",
    "draining_pipes": "Is there a pipe or drain outlet flowing into the stream? YES or NO.",
    "sewage_discharge": "Is there a visible discharge of dirty water, sewage or foam coming from a pipe? YES or NO.",
    "construction": "Are there construction works, machinery or building sites near the stream? YES or NO.",
    "vegetation_cuts": "Are there signs of recent vegetation cutting or mowing on the banks? YES or NO.",
}

def construire_prompt():
    lignes = [
        "You are helping a citizen fill in an urban stream observation form.",
        "Look at the photo and answer each field with ONE of the allowed options, or NOT_SURE.",
        "Answer NOT_SURE whenever the element is not clearly visible. Never guess.",
        "",
    ]
    for champ, question in QUESTIONS_VLM.items():
        options = FORMULAIRE[champ]["options"] + ["NOT_SURE"]
        lignes.append(f'- "{champ}": {question} Allowed: {", ".join(options)}.')
    lignes += [
        "- \"overall\": overall stream quality. GOOD = natural channel, riparian vegetation, clean water; "
        "MODERATE = some alterations; POOR = highly artificial, little vegetation or polluted. Allowed: GOOD, MODERATE, POOR.",
        "",
        "Reply with JSON only, no other text, in this exact format:",
        '{"field_name": {"value": "OPTION", "confidence": 0.0-1.0, "reason": "short reason"}}',
    ]
    return "\n".join(lignes)

PROMPT_VLM = construire_prompt()

def extraire_json(texte):
    trouve = re.search(r"\{.*\}", texte, re.DOTALL)
    if not trouve:
        return {}
    try:
        return json.loads(trouve.group(0))
    except json.JSONDecodeError:
        return {}

@torch.no_grad()
def interroger_vlm(img_pil, prompt=None, max_new_tokens=700):
    prompt = prompt or PROMPT_VLM
    messages = [{"role": "user", "content": [{"type": "image", "image": img_pil},
                                             {"type": "text", "text": prompt}]}]
    try:
        entrees = vlm_processor.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt").to(vlm.device)
    except Exception:
        # versions antérieures de transformers : image passée à part
        gabarit = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
        texte = vlm_processor.apply_chat_template(gabarit, add_generation_prompt=True, tokenize=False)
        entrees = vlm_processor(text=[texte], images=[img_pil], return_tensors="pt").to(vlm.device)

    sortie = vlm.generate(**entrees, max_new_tokens=max_new_tokens, do_sample=False)
    brut = vlm_processor.batch_decode(sortie[:, entrees["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    return extraire_json(brut), brut


In [8]:
PROMPT_ECUME = (
    "Look at the white patches on the water surface. Classify them as:\n"
    "- POLLUTION_FOAM: persistent foam floating on calm or slow water, often yellowish or grey, "
    "typical of detergents or sewage;\n"
    "- NATURAL_WHITEWATER: white water created by rapids, a waterfall, a weir or turbulence;\n"
    "- NOT_SURE if you cannot tell.\n"
    'Reply with JSON only: {\"value\": \"OPTION\", \"confidence\": 0.0-1.0, \"reason\": \"short reason\"}'
)

PROMPT_ANIMAL = (
    "Look carefully at the water. Can you identify the body of a dead animal, "
    "with visible body parts such as a head, an eye, fins, a tail, scales, "
    "feathers or fur?\n"
    "Answer YES only if such body parts are visible. "
    "Answer NO for any man-made object or if no animal body is visible.\n"
    'Reply with JSON only: {\"value\": \"YES|NO|NOT_SURE\", \"confidence\": 0.0-1.0, ' + \
    '\"reason\": \"short reason\"}'
)

def lire_reponse(brut, options, source):
    if not isinstance(brut, dict) or "value" not in brut:
        return pas_sur("réponse illisible", source)
    valeur = str(brut.get("value", "")).upper().strip()
    raison = str(brut.get("reason", ""))[:120]
    if valeur == "NOT_SURE":
        return pas_sur(raison or "le modèle n'a pas pu trancher", source)
    if valeur not in options:
        return pas_sur("réponse hors options", source)
    try:
        conf = min(float(brut.get("confidence", 0.5)), 0.8)
    except (TypeError, ValueError):
        conf = 0.5
    return reponse(valeur, conf, raison, source)

def arbitrer_ecume(img_pil):
    brut, _ = interroger_vlm(img_pil, PROMPT_ECUME, max_new_tokens=120)
    return lire_reponse(brut, ["POLLUTION_FOAM", "NATURAL_WHITEWATER"], "arbitrage")

def detecter_animal_mort(img_pil):
    brut, _ = interroger_vlm(img_pil, PROMPT_ANIMAL, max_new_tokens=120)
    r = lire_reponse(brut, ["YES", "NO"], "vlm")
    r["a_valider"] = r["valeur"] == "YES"   # jamais d'alerte sans validation humaine
    return r

print("prompts et lecture des réponses à jour")

prompts et lecture des réponses à jour


## 6. Fusion des sources

`normaliser_vlm` accepte maintenant `NOT_SURE`.

In [9]:
def normaliser_vlm(brut_vlm, champ):
    r = brut_vlm.get(champ)
    if not isinstance(r, dict):
        return None
    valeur = str(r.get("value", "NOT_SURE")).upper().strip()
    options = FORMULAIRE[champ]["options"] if champ in FORMULAIRE else ["GOOD", "MODERATE", "POOR"]
    if valeur == "NOT_SURE":
        return pas_sur(str(r.get("reason", ""))[:120] or "le modèle n'est pas sûr", "vlm")
    if valeur not in options:
        return pas_sur("réponse du modèle hors options", "vlm")
    try:
        conf = min(float(r.get("confidence", 0.5)), 0.8)
    except (TypeError, ValueError):
        conf = 0.5
    return reponse(valeur, conf, str(r.get("reason", ""))[:120], "vlm")

def fusionner(regle, vlm_r):
    candidats = [c for c in (regle, vlm_r) if c and c["valeur"] != "NOT_SURE"]
    if not candidats:
        return regle or vlm_r or pas_sur("aucune source")
    if len(candidats) == 2 and candidats[0]["valeur"] == candidats[1]["valeur"]:
        conf = min(1.0, max(c["confiance"] for c in candidats) + 0.15)
        return reponse(candidats[0]["valeur"], conf,
                       f"{candidats[0]['raison']} ; confirmé par le modèle", "regle+vlm")
    meilleur = max(candidats, key=lambda c: c["confiance"])
    if len(candidats) == 2:
        meilleur = dict(meilleur, raison=meilleur["raison"] + " (sources en désaccord)",
                        confiance=round(meilleur["confiance"] - 0.1, 2))
    return meilleur

def appliquer_seuil(r):
    if r["valeur"] != "NOT_SURE" and r["confiance"] < SEUIL_CONFIANCE:
        return dict(r, valeur="NOT_SURE", raison=f"confiance trop faible ({r['confiance']}) — {r['raison']}")
    return r

## 7. Déchets et évaluation globale

In [10]:
from ultralytics import YOLO

CHEMIN_DECHETS = f"{DOSSIERS['modeles']}/dechets/weights/best.pt"
modele_dechets = YOLO(CHEMIN_DECHETS) if os.path.exists(CHEMIN_DECHETS) else None
print("Détecteur de déchets :", "chargé" if modele_dechets else "absent (ignoré)")

def detecter_dechets(img_pil, seuil=0.4):
    if modele_dechets is None:
        return None
    boites = modele_dechets(img_pil, conf=seuil, verbose=False)[0].boxes
    return [{"classe": modele_dechets.names[int(b.cls)], "confiance": round(float(b.conf), 2),
             "boite": [int(v) for v in b.xyxy[0].tolist()]} for b in boites]

POINTS = {
    ("bank_type", "NAT"): (2, "berges naturelles"),
    ("bank_type", "LAYED_STONES"): (0, "berges en pierres posées"),
    ("bank_type", "ARTIFICIAL"): (-2, "berges artificielles"),
    ("bottom_type", "ARTIFICIAL"): (-1, "fond artificiel"),
    ("vegetation_left", "YES"): (1, "végétation rive gauche"),
    ("vegetation_right", "YES"): (1, "végétation rive droite"),
    ("vegetation_left", "NO"): (-1, "pas de végétation rive gauche"),
    ("vegetation_right", "NO"): (-1, "pas de végétation rive droite"),
    ("impervious_left", "YES"): (-1, "rive gauche imperméable"),
    ("impervious_right", "YES"): (-1, "rive droite imperméable"),
    ("water_aspect", "CLEAR"): (2, "eau claire"),
    ("water_aspect", "MUDDY"): (-1, "eau boueuse"),
    ("water_aspect", "FOAM"): (-2, "mousse en surface"),
    ("water_aspect", "ALTERED_COLOR"): (-2, "couleur de l'eau anormale"),
    ("barriers", "YES"): (-1, "obstacle dans le lit"),
    ("sewage_discharge", "YES"): (-3, "rejet d'eaux usées"),
    ("construction", "YES"): (-1, "travaux à proximité"),
}

def evaluer(formulaire, nb_dechets, avis_vlm, animal_mort=None):
    score, raisons = 0, []
    if animal_mort and animal_mort["valeur"] == "YES":
        score -= 3
        raisons.append("-3 animal mort signalé (à confirmer par un humain)")
    for champ, r in formulaire.items():
        if (champ, r["valeur"]) in POINTS:
            pts, texte = POINTS[(champ, r["valeur"])]
            score += pts
            raisons.append(f"{'+' if pts >= 0 else ''}{pts} {texte}")
    if nb_dechets:
        pts = -2 if nb_dechets >= 3 else -1
        score += pts
        raisons.append(f"{pts} {nb_dechets} déchet(s) détecté(s)")

    note_regles = "GOOD" if score >= 3 else "POOR" if score <= -2 else "MODERATE"
    note = note_regles
    accord = None
    if avis_vlm and avis_vlm["valeur"] in ("GOOD", "MODERATE", "POOR"):
        accord = avis_vlm["valeur"] == note_regles
        if not accord:
            raisons.append(f"le modèle vision-langage propose {avis_vlm['valeur']} : à vérifier par le citoyen")
    return {"valeur": note, "score": score, "raisons": raisons,
            "accord_modele": accord, "confiance": 0.8 if accord else 0.55}

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Détecteur de déchets : chargé


## 8. Image annotée

In [11]:
COULEURS_MASQUE = {EAU: (40, 120, 255), ARBRES: (20, 140, 40), HERBES: (120, 200, 60),
                   ARBUSTES: (70, 170, 50), IMPER: (220, 60, 60), SOL: (150, 110, 60)}

def annoter(img_pil, masque, orientation, dechets):
    image = np.array(img_pil).copy()
    calque = image.copy()
    for g, couleur in COULEURS_MASQUE.items():
        calque[masque == g] = couleur
    image = cv2.addWeighted(image, 0.6, calque, 0.4, 0)

    h, w = image.shape[:2]
    cv2.line(image, (w // 2, 0), (w // 2, h), (255, 255, 255), 2)
    gauche, droite = ("RIVE GAUCHE", "RIVE DROITE") if orientation == "aval" else ("RIVE DROITE", "RIVE GAUCHE")
    taille = max(0.5, w / 1400)
    for texte, x in [(gauche, 12), (droite, w // 2 + 12)]:
        cv2.putText(image, texte, (x, 34), cv2.FONT_HERSHEY_SIMPLEX, taille, (0, 0, 0), 5)
        cv2.putText(image, texte, (x, 34), cv2.FONT_HERSHEY_SIMPLEX, taille, (255, 255, 255), 2)

    for d in dechets or []:
        x1, y1, x2, y2 = d["boite"]
        cv2.rectangle(image, (x1, y1), (x2, y2), (255, 200, 0), 2)
        cv2.putText(image, d["classe"], (x1, max(y1 - 6, 14)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 200, 0), 2)
    return Image.fromarray(image)

## 9. Contexte (J5) : météo, Hub'Eau, score de risque

In [12]:
import requests

def meteo_contexte(lat, lon):
    """Pluie et température : 3 jours passés + 3 jours de prévision."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": "precipitation_sum,temperature_2m_max",
        "past_days": 3,
        "forecast_days": 4,   # aujourd'hui + 3 jours
        "timezone": "Europe/Paris",
    }
    try:
        r = requests.get(url, params=params, timeout=15)
        r.raise_for_status()
        d = r.json()["daily"]
    except Exception as e:
        return {"disponible": False, "erreur": str(e)}

    pluie = [p or 0 for p in d["precipitation_sum"]]
    temp = [t for t in d["temperature_2m_max"] if t is not None]
    # indices : 0-2 = 3 jours passés, 3 = aujourd'hui, 4-6 = 3 jours à venir
    return {
        "disponible": True,
        "pluie_72h_mm": round(sum(pluie[0:3]), 1),
        "pluie_aujourdhui_mm": round(pluie[3], 1),
        "pluie_prevue_3j_mm": round(sum(pluie[4:7]), 1),
        "temp_max_aujourdhui": d["temperature_2m_max"][3],
        "temp_max_3j": max(temp[3:7]) if temp else None,
    }


In [13]:
def val(fiche, champ):
    """Valeur d'un champ du formulaire, en liste (gère simple et multiple)."""
    v = fiche["formulaire"].get(champ, {}).get("valeur")
    return v if isinstance(v, list) else [v]

def niveau(score):
    return "ELEVE" if score >= 4 else "MODERE" if score >= 2 else "FAIBLE"

CONSIGNES = {
    "humains": {
        "FAIBLE": "Pas de danger visible. Évitez tout de même de boire l'eau.",
        "MODERE": "Évitez le contact avec l'eau et lavez-vous les mains après la visite.",
        "ELEVE":  "Ne touchez pas l'eau. Signalement transmis au gestionnaire pour vérification.",
    },
    "animaux": {
        "FAIBLE": "Pas de danger visible pour les animaux.",
        "MODERE": "Empêchez votre chien de boire ou de se baigner.",
        "ELEVE":  "Tenez les animaux en laisse, loin de l'eau. Signalement transmis au gestionnaire.",
    },
}

def evaluer_risque(fiche, meteo):
    h, a, raisons = 0, 0, []   # score humains, score animaux
    aspect = val(fiche, "water_aspect")
    chaud = meteo.get("disponible") and (meteo.get("temp_max_aujourdhui") or 0) >= 25
    pluie = meteo.get("pluie_72h_mm", 0) if meteo.get("disponible") else 0

    # --- Signaux de l'image (J1 à J4) ---
    if "FOAM" in aspect:
        h += 2; a += 2; raisons.append("mousse sur l'eau : pollution possible")
    if "ALTERED_COLOR" in aspect:
        h += 2; a += 2; raisons.append("couleur de l'eau anormale")
        if chaud:
            h += 1; a += 2; raisons.append("eau colorée + chaleur : cyanobactéries possibles, dangereuses pour les chiens")
    if "MUDDY" in aspect:
        h += 1; a += 1; raisons.append("eau trouble")
    if "YES" in val(fiche, "sewage_discharge"):
        h += 3; a += 3; raisons.append("rejet d'eaux usées visible")
    if "YES" in val(fiche, "draining_pipes"):
        h += 1; a += 1; raisons.append("canalisation de drainage visible")

    sig = fiche.get("signalements", {})
    if sig.get("animal_mort", {}).get("valeur") == "YES":
        h += 3; a += 3; raisons.append("animal mort signalé (à confirmer)")
    if sig.get("dechets"):
        h += 1; a += 1; raisons.append(f"{len(sig['dechets'])} déchet(s) détecté(s) : blessure ou ingestion")

    # --- Contexte météo ---
    if pluie >= 20:
        h += 2; a += 2; raisons.append(f"{pluie} mm de pluie en 72 h : ruissellement et débordements d'égouts probables")
    elif pluie >= 10:
        h += 1; a += 1; raisons.append(f"{pluie} mm de pluie en 72 h")

    flow = str(val(fiche, "water_flow")[0])
    if flow.startswith("FAS") and pluie >= 10:
        h += 1; raisons.append("courant rapide après la pluie : risque de chute ou d'emport")
    if flow == "STAGNANT" and chaud:
        a += 1; raisons.append("eau stagnante et chaude")

    nh, na = niveau(h), niveau(a)

    # --- Tendance à 3 jours (optionnelle) ---
    tendance = "inconnue"
    if meteo.get("disponible"):
        tendance = "en hausse" if meteo["pluie_prevue_3j_mm"] >= 20 else "stable"

    return {
        "risque_humains": {"niveau": nh, "score": h, "consigne": CONSIGNES["humains"][nh]},
        "risque_animaux": {"niveau": na, "score": a, "consigne": CONSIGNES["animaux"][na]},
        "raisons": raisons or ["aucun signal de risque détecté"],
        "tendance_3j": tendance,
        "meteo": meteo,
        "a_valider_gestionnaire": "ELEVE" in (nh, na),  # jamais d'alerte directe
    }

In [14]:
from datetime import date, timedelta
from math import radians, sin, cos, asin, sqrt

HUBEAU_URL = "https://hubeau.eaufrance.fr/api"

def hubeau(chemin, params):
    """Essaie l'API v2 puis v1 ; renvoie la liste 'data' ou None."""
    for v in ("v2", "v1"):
        try:
            r = requests.get(f"{HUBEAU_URL}/{v}/hydrometrie/{chemin}", params=params, timeout=20)
            if r.status_code in (200, 206):
                return r.json().get("data", [])
        except Exception:
            pass
    return None

def distance_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(radians, (lat1, lon1, lat2, lon2))
    a = sin((lat2-lat1)/2)**2 + cos(lat1)*cos(lat2)*sin((lon2-lon1)/2)**2
    return 6371 * 2 * asin(sqrt(a))

def hubeau_contexte(lat, lon, rayon_km=10):
    stations = hubeau("referentiel/stations",
                      {"latitude": lat, "longitude": lon, "distance": rayon_km, "size": 20})
    if not stations:
        return {"disponible": False, "raison": f"aucune station à moins de {rayon_km} km"}

    # Trier par distance, garder la première station qui mesure un débit
    stations.sort(key=lambda s: distance_km(lat, lon, s["latitude_station"], s["longitude_station"]))
    for s in stations:
        code = s["code_station"]
        obs = hubeau("observations_tr", {"code_entite": code, "grandeur_hydro": "Q",
                                         "size": 1, "sort": "desc"})
        if not obs:
            continue
        debit = obs[0]["resultat_obs"] / 1000   # l/s → m³/s

        # Référence : débits moyens journaliers des 30 derniers jours
        elab = hubeau("obs_elab", {"code_entite": code, "grandeur_hydro_elab": "QmnJ",
                                   "date_debut_obs_elab": str(date.today() - timedelta(days=30)),
                                   "size": 40}) or []
        valeurs = sorted(e["resultat_obs_elab"] / 1000 for e in elab if e.get("resultat_obs_elab"))
        mediane = valeurs[len(valeurs)//2] if valeurs else None

        return {
            "disponible": True,
            "station": s.get("libelle_station"),
            "code_station": code,
            "distance_km": round(distance_km(lat, lon, s["latitude_station"], s["longitude_station"]), 1),
            "debit_m3s": round(debit, 2),
            "debit_median_30j_m3s": round(mediane, 2) if mediane else None,
            "ratio": round(debit / mediane, 2) if mediane else None,
        }
    return {"disponible": False, "raison": "stations proches sans mesure de débit"}


In [15]:
def appliquer_debit(r, hub):
    """Ajoute les règles Hub'Eau à un résultat de evaluer_risque."""
    h, a = r["risque_humains"]["score"], r["risque_animaux"]["score"]
    ratio = hub.get("ratio") if hub.get("disponible") else None

    if ratio is not None:
        if ratio >= 2:
            h += 2; a += 1
            r["raisons"].append(f"débit {ratio} fois supérieur à la normale : crue, courant dangereux")
        elif ratio >= 1.5:
            h += 1
            r["raisons"].append(f"débit élevé ({ratio} fois la normale)")
        elif ratio <= 0.3:
            a += 1
            r["raisons"].append(f"débit très faible ({ratio} fois la normale) : pollution plus concentrée")

    nh, na = niveau(h), niveau(a)
    r["risque_humains"] = {"niveau": nh, "score": h, "consigne": CONSIGNES["humains"][nh]}
    r["risque_animaux"] = {"niveau": na, "score": a, "consigne": CONSIGNES["animaux"][na]}
    r["a_valider_gestionnaire"] = "ELEVE" in (nh, na)
    r["hubeau"] = hub
    if r["raisons"][0] == "aucun signal de risque détecté" and len(r["raisons"]) > 1:
        r["raisons"].pop(0)
    return r

def evaluer_risque_complet(fiche, lat, lon, meteo=None, hub=None):
    meteo = meteo or meteo_contexte(lat, lon)
    hub = hub or hubeau_contexte(lat, lon)
    return appliquer_debit(evaluer_risque(fiche, meteo), hub)


## 10. Pipeline complet : une photo → fiche + risque

Même logique que `prefill` (J4) suivi de `evaluer_risque_complet` (J5), avec une
fonction `etape` qui indique à l'interface où en est l'analyse.

In [16]:
DOSSIER_API = "/content/api"
for d in ("photos", "annotees"):
    os.makedirs(f"{DOSSIER_API}/{d}", exist_ok=True)

LAT_DEFAUT, LON_DEFAUT = 48.8566, 2.3522   # Seine à Paris, si le téléphone n'envoie pas de GPS

def analyse_complete(chemin, orientation="aval", lat=LAT_DEFAUT, lon=LON_DEFAUT,
                     nom=None, etape=lambda texte: None):
    nom = nom or os.path.basename(chemin)

    etape("Segmentation de la scène")
    img = charger(chemin)
    masque = segmenter(img)
    if (masque == EAU).mean() < 0.05:
        fiche = {"photo": nom, "valide": False,
                 "message": "Cours d'eau pas assez visible : cadrez l'eau et les berges, puis reprenez la photo."}
        return {"fiche": fiche, "risque": None, "image_annotee_b64": None}

    regles = regles_berges(masque, orientation)
    analyse = analyser_eau(img, masque)
    regles["water_aspect"] = regle_eau(analyse)

    etape("Lecture de la photo par le modèle vision-langage")
    brut_vlm, _ = interroger_vlm(img)
    formulaire = {}
    for champ in FORMULAIRE:
        r_regle = regles.get(champ)
        r_vlm = normaliser_vlm(brut_vlm, champ) if champ in QUESTIONS_VLM else None
        formulaire[champ] = appliquer_seuil(fusionner(r_regle, r_vlm))

    arbitrages = {}
    if formulaire["water_aspect"]["valeur"] == "FOAM":
        etape("Arbitrage : écume naturelle ou mousse de pollution ?")
        arb = arbitrer_ecume(img)
        arbitrages["ecume_ou_mousse"] = arb
        if arb["valeur"] == "NATURAL_WHITEWATER" and arb["confiance"] >= 0.5:
            nouveau = regle_eau(dict(analyse, mousse=0.0))
            formulaire["water_aspect"] = appliquer_seuil(dict(
                nouveau, source="regle+arbitrage",
                raison=nouveau["raison"] + " ; blanc = écume naturelle selon le modèle"))
        elif arb["valeur"] == "NOT_SURE":
            formulaire["water_aspect"] = pas_sur("mousse ou écume naturelle : à confirmer", "arbitrage")

    etape("Recherche de déchets et d'animaux morts")
    animal = detecter_animal_mort(img)
    dechets = detecter_dechets(img)
    nb_dechets = len(dechets) if dechets is not None else None
    evaluation = evaluer(formulaire, nb_dechets, normaliser_vlm(brut_vlm, "overall"), animal)

    etape("Annotation de l'image")
    chemin_annote = f"{DOSSIER_API}/annotees/{os.path.splitext(os.path.basename(chemin))[0]}.jpg"
    annoter(img, masque, orientation, dechets).save(chemin_annote, quality=88)

    remplis = sum(r["valeur"] != "NOT_SURE" for r in formulaire.values())
    fiche = {
        "photo": nom,
        "valide": True,
        "orientation": orientation,
        "formulaire": formulaire,
        "overall_assessment": evaluation,
        "arbitrages": arbitrages,
        "signalements": {"animal_mort": animal, "dechets": dechets},
        "image_annotee": chemin_annote,
        "laisses_au_citoyen": LAISSES_AU_CITOYEN,
        "taux_prefill": round(remplis / len(FORMULAIRE), 2),
    }

    etape("Météo et débit de la rivière")
    risque = evaluer_risque_complet(fiche, lat, lon)
    risque["photo"] = nom

    with open(chemin_annote, "rb") as f:
        image_b64 = base64.b64encode(f.read()).decode()
    return {"fiche": fiche, "risque": risque, "image_annotee_b64": image_b64}


### Test sur une photo du Drive, avant d'ouvrir l'API

In [17]:
import torch
print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Carte :", torch.cuda.get_device_name(0))
    print(f"Mémoire GPU utilisée : {torch.cuda.memory_allocated() / 1e9:.1f} Go")

# Sur quel appareil chaque modèle est-il chargé ?
for nom, obj in list(globals().items()):
    if hasattr(obj, "parameters") and callable(getattr(obj, "parameters")):
        try:
            p = next(obj.parameters())
            print(f"{nom:20s} \u2192 {p.device}, {p.dtype}")
        except Exception:
            pass

import transformers
print("transformers :", transformers.__version__)

GPU disponible : True
Carte : Tesla T4
Mémoire GPU utilisée : 7.6 Go
seg_model            → cuda:0, torch.float32
vlm                  → cuda:0, torch.float16
modele_dechets       → cpu, torch.float32
transformers : 5.16.1


In [18]:
debut = time.time()
test = analyse_complete(photos[0], etape=lambda t: print("  ->", t))
f, r = test["fiche"], test["risque"]
print(f"\n{f['photo']} : {time.time() - debut:.0f} s")
if f["valide"]:
    print(f"pré-rempli {f['taux_prefill']:.0%} | global {f['overall_assessment']['valeur']} | "
          f"humains {r['risque_humains']['niveau']} | animaux {r['risque_animaux']['niveau']}")
    print("Mentions 'hors options' :", json.dumps(f, ensure_ascii=False).count("hors options"))
else:
    print(f["message"])


  -> Segmentation de la scène
  -> Lecture de la photo par le modèle vision-langage
  -> Recherche de déchets et d'animaux morts
  -> Annotation de l'image
  -> Météo et débit de la rivière

berge_betonnee_00.jpg : 51 s
pré-rempli 56% | global POOR | humains FAIBLE | animaux FAIBLE
Mentions 'hors options' : 0


## 11. L'API

- `GET /sante` : vérifie que l'API répond.
- `POST /analyser` : reçoit la photo (+ `orientation`, `lat`, `lon`) et renvoie tout de suite un `id`.
- `GET /resultat/{id}` : état de l'analyse (`en_attente`, `en_cours`, `termine`, `erreur`), étape en cours, puis le résultat.

L'analyse prend 1 à 2 minutes : l'interface interroge `/resultat` toutes les quelques
secondes au lieu d'attendre une seule réponse, que le tunnel couperait au bout de 100 s.
Un verrou empêche deux analyses de se lancer en même temps sur le GPU.

In [19]:
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
import uvicorn

app = FastAPI(title="StreamSentinel API")
TACHES = {}
VERROU_GPU = threading.Lock()

def executer(id_tache, chemin, orientation, lat, lon, nom):
    t = TACHES[id_tache]
    def etape(texte):
        t["etape"] = texte
    try:
        with VERROU_GPU:
            t["etat"] = "en_cours"
            t["resultat"] = analyse_complete(chemin, orientation, lat, lon, nom, etape)
        t["etat"] = "termine"
        t["etape"] = "Analyse terminée"
    except Exception as e:
        t["etat"] = "erreur"
        t["erreur"] = f"{type(e).__name__} : {e}"
    t["duree_s"] = round(time.time() - t["debut"], 1)

@app.get("/sante")
def sante():
    en_cours = sum(t["etat"] in ("en_attente", "en_cours") for t in TACHES.values())
    return {"ok": True, "gpu": torch.cuda.is_available(), "analyses_en_cours": en_cours}

@app.post("/analyser")
async def analyser(photo: UploadFile = File(...), orientation: str = Form("aval"),
                   lat: float = Form(LAT_DEFAUT), lon: float = Form(LON_DEFAUT)):
    if orientation not in ("amont", "aval"):
        raise HTTPException(400, "orientation doit valoir 'amont' ou 'aval'")
    id_tache = uuid.uuid4().hex[:8]
    ext = os.path.splitext(photo.filename or "")[1].lower() or ".jpg"
    chemin = f"{DOSSIER_API}/photos/{id_tache}{ext}"
    with open(chemin, "wb") as f:
        f.write(await photo.read())
    TACHES[id_tache] = {"etat": "en_attente", "etape": "En file d'attente", "debut": time.time()}
    threading.Thread(target=executer, daemon=True,
                     args=(id_tache, chemin, orientation, lat, lon, photo.filename or id_tache + ext)).start()
    return {"id": id_tache}

@app.get("/resultat/{id_tache}")
def resultat(id_tache: str):
    t = TACHES.get(id_tache)
    if t is None:
        raise HTTPException(404, "analyse inconnue (la session Colab a peut-être redémarré)")
    reponse_api = {k: v for k, v in t.items() if k != "debut"}
    reponse_api["ecoule_s"] = round(time.time() - t["debut"])
    return reponse_api

# (re)démarrage du serveur en arrière-plan
if "serveur" in globals():
    serveur.should_exit = True
    time.sleep(2)
serveur = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=serveur.run, daemon=True).start()
time.sleep(3)

import requests
print("Test local :", requests.get("http://localhost:8000/sante").json())


Test local : {'ok': True, 'gpu': True, 'analyses_en_cours': 0}


## 12. Le tunnel : rendre l'API accessible depuis ton Mac

Cloudflare fournit gratuitement une adresse temporaire, sans compte.
**Elle change à chaque exécution de cette cellule** : recopie-la dans Streamlit à chaque fois.

In [20]:
!pip install -q pyngrok
from pyngrok import ngrok
from google.colab import userdata

DOMAINE = "directed-freight-gully.ngrok-free.dev"   # à remplacer par ton domaine ngrok

ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
for t in ngrok.get_tunnels():            # ferme un éventuel ancien tunnel
    ngrok.disconnect(t.public_url)

URL_API = ngrok.connect("127.0.0.1:8000", domain=DOMAINE).public_url
print("Adresse de l'API (fixe) :", URL_API)
print("Test :", URL_API + "/sante")

Adresse de l'API (fixe) : https://directed-freight-gully.ngrok-free.dev
Test : https://directed-freight-gully.ngrok-free.dev/sante


In [21]:
# ===== J5 bis : chaque règle du risque rattachée à sa source =====
# Les fiches OneAquaHealth (Schmeller et al., 2026, doi:10.5281/zenodo.20345207, CC-BY 4.0)
# justifient POURQUOI un signal compte pour la santé ; elles ne donnent AUCUN seuil
# chiffré pour des observations photo. Les points et les seuils restent donc des
# choix de l'équipe, affichés comme tels. Les scores sont identiques à la version précédente.
SOURCES_OAH = {
    "I":    "OAH factsheet I (diatoms): harmful algal blooms produce toxins, kill fish and threaten human health",
    "III":  "OAH factsheet III (fish): fish can carry pathogens transmissible to humans",
    "VII":  "OAH factsheet VII (fecal coliforms): waste inputs mean exposure to enteric pathogens",
    "VIII": "OAH factsheet VIII (pathogens): abundance rises with wastewater discharge and runoff",
    "IX":   "OAH factsheet IX (antibiotic resistance genes): spread through wastewater and urban runoff",
    "EQUIPE": "Team assumption: not covered by the OAH factsheets",
}

def _regle(detail, raisons, texte, dh, da, sources, label_en):
    detail.append({"texte": texte, "label": label_en, "humains": dh, "animaux": da,
                   "sources": [SOURCES_OAH[s] for s in sources]})
    raisons.append(texte)
    return dh, da

def evaluer_risque(fiche, meteo):
    h, a, raisons, detail = 0, 0, [], []
    aspect = val(fiche, "water_aspect")
    chaud = meteo.get("disponible") and (meteo.get("temp_max_aujourdhui") or 0) >= 25
    pluie = meteo.get("pluie_72h_mm", 0) if meteo.get("disponible") else 0

    def r(texte, dh, da, sources, label):
        nonlocal h, a
        x, y = _regle(detail, raisons, texte, dh, da, sources, label)
        h += x; a += y

    if "FOAM" in aspect:
        r("mousse sur l'eau : pollution possible", 2, 2, ["EQUIPE"], "Foam on the water: possible pollution")
    if "ALTERED_COLOR" in aspect:
        r("couleur de l'eau anormale", 2, 2, ["I"], "Abnormal water colour: possible algal bloom")
        if chaud:
            r("eau colorée + chaleur : cyanobactéries possibles, dangereuses pour les chiens", 1, 2,
              ["EQUIPE"], "Coloured water and heat (25 °C or more): bloom more likely")
    if "MUDDY" in aspect:
        r("eau trouble", 1, 1, ["EQUIPE"], "Muddy water")
    if "YES" in val(fiche, "sewage_discharge"):
        r("rejet d'eaux usées visible", 3, 3, ["VII", "VIII", "IX"], "Visible sewage discharge")
    if "YES" in val(fiche, "draining_pipes"):
        r("canalisation de drainage visible", 1, 1, ["IX"], "Visible drainage pipe (urban runoff)")

    sig = fiche.get("signalements", {})
    if sig.get("animal_mort", {}).get("valeur") == "YES":
        r("animal mort signalé (à confirmer)", 3, 3, ["III", "I"], "Dead animal reported (to be confirmed)")
    if sig.get("dechets"):
        n = len(sig["dechets"])
        r(f"{n} déchet(s) détecté(s) : blessure ou ingestion", 1, 1, ["EQUIPE"],
          f"{n} litter item(s): injury or ingestion")

    if pluie >= 20:
        r(f"{pluie} mm de pluie en 72 h : ruissellement et débordements d'égouts probables", 2, 2,
          ["VIII", "IX"], f"{pluie} mm of rain in 72 h: runoff and sewer overflows (threshold: team)")
    elif pluie >= 10:
        r(f"{pluie} mm de pluie en 72 h", 1, 1, ["VIII", "IX"], f"{pluie} mm of rain in 72 h (threshold: team)")

    flow = str(val(fiche, "water_flow")[0])
    if flow.startswith("FAS") and pluie >= 10:
        r("courant rapide après la pluie : risque de chute ou d'emport", 1, 0, ["EQUIPE"],
          "Fast current after rain: risk of falling in")
    if flow == "STAGNANT" and chaud:
        r("eau stagnante et chaude", 0, 1, ["EQUIPE"], "Stagnant, warm water")

    nh, na = niveau(h), niveau(a)
    tendance = "inconnue"
    if meteo.get("disponible"):
        tendance = "en hausse" if meteo["pluie_prevue_3j_mm"] >= 20 else "stable"
    return {
        "risque_humains": {"niveau": nh, "score": h, "consigne": CONSIGNES["humains"][nh]},
        "risque_animaux": {"niveau": na, "score": a, "consigne": CONSIGNES["animaux"][na]},
        "raisons": raisons or ["aucun signal de risque détecté"],
        "regles": detail,
        "tendance_3j": tendance,
        "meteo": meteo,
        "a_valider_gestionnaire": "ELEVE" in (nh, na),
    }

def appliquer_debit(r, hub):
    h, a = r["risque_humains"]["score"], r["risque_animaux"]["score"]
    ratio = hub.get("ratio") if hub.get("disponible") else None
    regle = None
    if ratio is not None:
        if ratio >= 2:
            regle = (f"débit {ratio} fois supérieur à la normale : crue, courant dangereux", 2, 1,
                     f"River flow {ratio}× the 30-day median: flood, dangerous current")
        elif ratio >= 1.5:
            regle = (f"débit élevé ({ratio} fois la normale)", 1, 0, f"High river flow ({ratio}× normal)")
        elif ratio <= 0.3:
            regle = (f"débit très faible ({ratio} fois la normale) : pollution plus concentrée", 0, 1,
                     f"Very low river flow ({ratio}× normal): pollution more concentrated")
    if regle:
        texte, dh, da, label = regle
        r.setdefault("regles", [])
        _regle(r["regles"], r["raisons"], texte, dh, da, ["EQUIPE"], label)
        h += dh; a += da

    nh, na = niveau(h), niveau(a)
    r["risque_humains"] = {"niveau": nh, "score": h, "consigne": CONSIGNES["humains"][nh]}
    r["risque_animaux"] = {"niveau": na, "score": a, "consigne": CONSIGNES["animaux"][na]}
    r["a_valider_gestionnaire"] = "ELEVE" in (nh, na)
    r["hubeau"] = hub
    if r["raisons"][0] == "aucun signal de risque détecté" and len(r["raisons"]) > 1:
        r["raisons"].pop(0)
    return r

# Vérification : même score qu'avant sur un cas complet
_test = evaluer_risque(
    {"formulaire": {"water_aspect": {"valeur": "ALTERED_COLOR"}, "sewage_discharge": {"valeur": "YES"},
                    "draining_pipes": {"valeur": "NO"}, "water_flow": {"valeur": "STAGNANT"}},
     "signalements": {"dechets": [1, 2]}},
    {"disponible": True, "temp_max_aujourdhui": 28, "pluie_72h_mm": 12, "pluie_prevue_3j_mm": 0})
print("humains", _test["risque_humains"]["score"], "| animaux", _test["risque_animaux"]["score"])
for g in _test["regles"]:
    print(f"  +{g['humains']}/+{g['animaux']}  {g['label']}  <- {g['sources'][0][:40]}")

humains 8 | animaux 10
  +2/+2  Abnormal water colour: possible algal bloom  <- OAH factsheet I (diatoms): harmful algal
  +1/+2  Coloured water and heat (25 °C or more): bloom more likely  <- Team assumption: not covered by the OAH 
  +3/+3  Visible sewage discharge  <- OAH factsheet VII (fecal coliforms): was
  +1/+1  2 litter item(s): injury or ingestion  <- Team assumption: not covered by the OAH 
  +1/+1  12 mm of rain in 72 h (threshold: team)  <- OAH factsheet VIII (pathogens): abundanc
  +0/+1  Stagnant, warm water  <- Team assumption: not covered by the OAH 
